# BÀI TẬP LỚN 02: DỰ ĐOÁN GIÁ BẤT ĐỘNG SẢN BẰNG MẠNG NƠ-RON HỒI QUY 5 TẦNG
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Tiền xử lý tập dữ liệu hồi quy bất động sản House Price (chuẩn hóa z-score đặc trưng và mục tiêu).
2. Xây dựng mạng 5 tầng MLP hồi quy (293 -> 128 -> 64 -> 32 -> 16 -> 1).
3. Tối ưu hóa hàm mất mát MSE Loss bằng giải thuật Adam.
4. Đánh giá sai số hồi quy trên các chỉ số MAE, MSE, RMSE, R² Score và vẽ Scatter Plot.


In [ ]:
# Cell [1]: Nap du lieu va Tien xu ly
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

candidate_paths = [
    '../data/house_price.csv',
    'data/house_price.csv',
    '../../data/house_price.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

df_h = pd.read_csv(data_path)
print(f"[INFO] Nap du lieu thanh cong: {df_h.shape}")

X = df_h.drop('SalePrice', axis=1).values
y = df_h['SalePrice'].values

scaler_X = StandardScaler()
X_norm = scaler_X.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")


In [ ]:
# Cell [2]: Dinh nghia mo hinh Hoi quy 5 tang MLP (293 -> 1)
class HousePriceMLP(nn.Module):
    def __init__(self, in_features=293):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(),
            nn.Linear(128, 64),          nn.ReLU(),
            nn.Linear(64, 32),           nn.ReLU(),
            nn.Linear(32, 16),           nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model_h = HousePriceMLP(in_features=X_train.shape[1])
print(model_h)


In [ ]:
# Cell [3]: Huan luyen mo hinh voi MSELoss va Adam Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model_h.parameters(), lr=0.01)

X_tr_t = torch.FloatTensor(X_train)
y_tr_t = torch.FloatTensor(y_train)
X_te_t = torch.FloatTensor(X_test)

losses = []
for epoch in range(1, 26):
    model_h.train()
    optimizer.zero_grad()
    preds = model_h(X_tr_t)
    loss = criterion(preds, y_tr_t)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/25] - MSE Loss: {loss.item():.2e}")

print("[OK] Huan luyen hoi quy hoan tat!")


In [ ]:
# Cell [4]: Danh gia sai so Hoi quy va Ve do thi Scatter Plot
model_h.eval()
with torch.no_grad():
    y_pred = model_h(X_te_t).numpy()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=== KET QUA HOI QUY GIA NHA DUC MLP ===")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R2:   {r2:.4f}")

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='royalblue', edgecolors='k', s=30)
min_v = min(y_test.min(), y_pred.min())
max_v = max(y_test.max(), y_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Ideal y = y_pred')
plt.title(f'House Price Prediction (R2 = {r2:.4f})')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.legend()
plt.tight_layout()
plt.show()
